# Marketing Campaign Analysis — Data Exploration

**Notebook 01 di 03** · Esplorazione iniziale del dataset

## Obiettivo del progetto
Rispondere alla domanda: *"Come dovrebbe un'azienda food & beverage ottimizzare l'allocazione del budget delle prossime campagne marketing, basandosi sul comportamento storico dei suoi clienti?"*

## Obiettivo di questo notebook
Esplorare la struttura del dataset, comprendere le variabili disponibili, identificare problemi di qualità dei dati (missing values, outlier, inconsistenze) e definire le prime ipotesi di analisi.

## Dataset
**Customer Personality Analysis** — 2.240 clienti di un'azienda food & beverage, con dati demografici, storico di spesa su 6 categorie di prodotti, utilizzo di 3 canali di vendita e risposta a 6 campagne marketing precedenti.

Fonte: [Kaggle — Customer Personality Analysis](https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis)

---

In [1]:
# Librerie di data manipulation
import pandas as pd
import numpy as np

# Librerie di visualizzazione
import matplotlib.pyplot as plt
import seaborn as sns

# Configurazioni globali
pd.set_option('display.max_columns', None)  # Mostra tutte le colonne quando stampiamo un DataFrame
pd.set_option('display.width', 200)         # Amplia la larghezza di stampa per leggibilità

# Stile grafici
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)    # Dimensione default dei grafici

print("✅ Librerie importate correttamente")
print(f"   pandas: {pd.__version__}")
print(f"   numpy:  {np.__version__}")

✅ Librerie importate correttamente
   pandas: 3.0.2
   numpy:  2.4.3


## 1. Caricamento del dataset

Il dataset è in formato CSV, con separatore tab (`\t`) — non la virgola standard. È una peculiarità che va gestita al momento del caricamento.

In [2]:
# Path al file CSV — relativo alla posizione del notebook (notebooks/)
DATA_PATH = r'C:\Users\rober\OneDrive\Desktop\Portfolio_Projects\01-marketing-analytics\data\raw\marketing_campaign.csv'

# Il file usa tab come separatore, non virgola
df = pd.read_csv(DATA_PATH, sep='\t')

print(f"✅ Dataset caricato correttamente")
print(f"   Righe:    {df.shape[0]:,}")
print(f"   Colonne:  {df.shape[1]}")

✅ Dataset caricato correttamente
   Righe:    2,240
   Colonne:  29


## 2. Prima ispezione: le prime righe

Vediamo com'è fatto il dataset guardando le prime 5 righe. Questo ci dà un'idea iniziale delle colonne disponibili e dei tipi di valore.

In [3]:
df.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0


## 3. Struttura del dataset: tipi di dato e valori mancanti

Prima di analizzare i dati, dobbiamo capire:
- **Che tipo di valori contengono le colonne** (numerici, categorici, date)
- **Se ci sono missing values** e dove si concentrano
- **Se ci sono colonne inutili** (costanti o ridondanti)

Questa fase è fondamentale: eventuali problemi di qualità dei dati vanno identificati ora, prima di costruire qualsiasi analisi.

In [4]:
# Panoramica completa: tipi di dato, non-null count per colonna, memoria
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   str    
 3   Marital_Status       2240 non-null   str    
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   str    
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   int64  
 16 

### 📌 Osservazioni da `df.info()`

- **Tipi di dato**: la maggior parte delle colonne è numerica (`int64` / `float64`), 3 colonne sono di tipo `object` (stringhe): `Education`, `Marital_Status`, `Dt_Customer`. Quest'ultima rappresenta una data e dovrà essere convertita in `datetime` per poter essere manipolata come tale.
- **Missing values**: la colonna `Income` presenta valori mancanti (meno del 2% del dataset) — la gestiremo nella fase di data cleaning.
- **Memoria**: il dataset è molto piccolo (~500 KB in RAM), nessuna preoccupazione di performance.

In [5]:
# Conteggio missing values per colonna — mostra solo le colonne problematiche
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) == 0:
    print("✅ Nessun missing value nel dataset")
else:
    print(f"⚠️  Missing values trovati in {len(missing)} colonne:\n")
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
    missing_df = pd.DataFrame({
        'Missing count': missing,
        'Missing %': missing_pct[missing.index]
    })
    print(missing_df)

⚠️  Missing values trovati in 1 colonne:

        Missing count  Missing %
Income             24       1.07


In [6]:
# Verifica: ci sono colonne costanti (stesso valore in tutte le righe)?
constant_cols = [col for col in df.columns if df[col].nunique() == 1]

if constant_cols:
    print(f"⚠️  Trovate {len(constant_cols)} colonne costanti (nessuna variabilità):")
    for col in constant_cols:
        print(f"   • {col}: valore unico = {df[col].iloc[0]}")
else:
    print("✅ Nessuna colonna costante")

⚠️  Trovate 2 colonne costanti (nessuna variabilità):
   • Z_CostContact: valore unico = 3
   • Z_Revenue: valore unico = 11


### 📌 Colonne costanti: `Z_CostContact` e `Z_Revenue`

Le due colonne `Z_CostContact` e `Z_Revenue` contengono lo stesso valore per tutti i 2.240 clienti. Sono quindi **prive di valore analitico** e verranno rimosse nella fase di data cleaning. Si tratta probabilmente di metadati residuati dal dataset originale (pubblicato per un business case iFood).

In [7]:
# Statistiche descrittive per le colonne numeriche
df.describe().T  # .T = transpose, una colonna per statistica invece che per variabile

,count,mean,std,min,25%,50%,75%,max
ID,2240.0,5592.159821,3246.662198,0.0,2828.25,5458.5,8427.75,11191.0
Year_Birth,2240.0,1968.805804,11.984069,1893.0,1959.00,1970.0,1977.00,1996.0
Income,2216.0,52247.251354,25173.076661,1730.0,35303.00,51381.5,68522.00,666666.0
Kidhome,2240.0,0.444196,0.538398,0.0,0.00,0.0,1.00,2.0
Teenhome,2240.0,0.506250,0.544538,0.0,0.00,0.0,1.00,2.0
Recency,2240.0,49.109375,28.962453,0.0,24.00,49.0,74.00,99.0
MntWines,2240.0,303.935714,336.597393,0.0,23.75,173.5,504.25,1493.0
MntFruits,2240.0,26.302232,39.773434,0.0,1.00,8.0,33.00,199.0
MntMeatProducts,2240.0,166.950000,225.715373,0.0,16.00,67.0,232.00,1725.0
MntFishProducts,2240.0,37.525446,54.628979,0.0,3.00,12.0,50.00,259.0


### 📌 Osservazioni dalle statistiche descrittive

L'ispezione di `df.describe()` rivela alcuni elementi che richiederanno attenzione nella fase di cleaning:

**🎂 Anno di nascita sospetto**
- `Year_Birth` ha un minimo di **1893**, che implicherebbe clienti di oltre 130 anni. Sono con ogni probabilità errori di data entry o valori di test. Da verificare e trattare.

**💰 Reddito estremo**
- `Income` presenta un massimo di **666.666 €**, contro una mediana di circa 51.000 €. Il valore è così specifico da apparire sintetico. Un singolo outlier isolato può distorcere medie e modelli — da verificare.

**🍷 Spese elevate ma plausibili**
- `MntWines` raggiunge 1.493 € e `MntMeatProducts` 1.725 € in 2 anni. Sono valori alti ma non implausibili per clienti premium. Non sono outlier patologici, ma potrebbero identificare un segmento "high-value" di grande interesse strategico per le raccomandazioni finali.

**📦 Canali di acquisto**
- Le variabili `Num*Purchases` hanno distribuzioni molto diverse tra loro: analizzeremo in dettaglio le differenze nella fase EDA.

## 4. Zoom sugli outlier sospetti

Prima di chiudere l'esplorazione, verifichiamo nel dettaglio i due punti che hanno destato sospetto: i `Year_Birth` anomali e il reddito da 666.666 €.

In [8]:
# Clienti nati prima del 1940 (oltre 85 anni circa — soglia ragionevole)
old_customers = df[df['Year_Birth'] < 1940].sort_values('Year_Birth')
print(f"Clienti con Year_Birth < 1940: {len(old_customers)}")
old_customers[['ID', 'Year_Birth', 'Income', 'Education', 'Marital_Status']]

Clienti con Year_Birth < 1940: 3


,ID,Year_Birth,Income,Education,Marital_Status
239,11004,1893,60182.0,2n Cycle,Single
339,1150,1899,83532.0,PhD,Together
192,7829,1900,36640.0,2n Cycle,Divorced


In [9]:
# Top 5 redditi più alti
print("Top 5 redditi più alti nel dataset:\n")
df.nlargest(5, 'Income')[['ID', 'Year_Birth', 'Income', 'Education', 'Marital_Status']]

Top 5 redditi più alti nel dataset:



,ID,Year_Birth,Income,Education,Marital_Status
2233,9432,1977,666666.0,Graduation,Together
617,1503,1976,162397.0,PhD,Together
687,1501,1982,160803.0,PhD,Married
1300,5336,1971,157733.0,Master,Together
164,8475,1973,157243.0,PhD,Married


## 5. Sintesi del Data Exploration

Il dataset **Customer Personality Analysis** contiene 2.240 clienti e 29 variabili suddivise in 6 famiglie logiche: identificativi, demografia, spesa per categoria di prodotto, canali di acquisto, risposta alle campagne marketing, metriche di servizio.

### Problemi di qualità identificati
| Problema | Colonna/e | Gravità | Azione prevista |
|---|---|---|---|
| Missing values | `Income` (24 record, 1.1%) | Bassa | Imputazione o rimozione |
| Colonne costanti | `Z_CostContact`, `Z_Revenue` | Nessun valore analitico | Rimozione |
| Outlier anagrafici | `Year_Birth` < 1940 | Da verificare | Rimozione se confermati errori |
| Outlier reddito | `Income` = 666.666 | Da verificare | Rimozione se singoletto isolato |
| Formato data | `Dt_Customer` è stringa | Bassa | Conversione a `datetime` |

### Opportunità analitiche identificate
- **Segmentazione clienti**: possibile costruire segmenti basati su demografia (età, reddito, situazione familiare) × comportamento di spesa × canali preferiti.
- **Total spending**: utile creare una feature aggregata `MntTotal` per analisi di Customer Lifetime Value semplificato.
- **Efficacia campagne**: le 6 variabili `AcceptedCmp*` + `Response` permettono analisi comparativa del ROI delle diverse campagne per segmento.
- **Customer tenure**: `Dt_Customer` permette di calcolare da quanto un cliente è attivo, variabile probabilmente correlata al comportamento di spesa.

### Prossimo step (Notebook 02 — Data Cleaning & Feature Engineering)
1. Gestire i missing values in `Income`
2. Rimuovere colonne costanti e outlier confermati
3. Convertire `Dt_Customer` in formato data e creare la feature `Customer_Tenure_Days`
4. Creare feature aggregate: `Age`, `MntTotal`, `TotalPurchases`, `TotalCampaignsAccepted`
5. Semplificare variabili categoriche (`Marital_Status` ha categorie ridondanti da consolidare)

In [10]:
# Salvataggio di una copia del dataset caricato (per riferimento rapido nei notebook successivi)
# Questo non modifica nulla, è solo una conferma finale
print(f"Notebook 01 — Data Exploration completato.")
print(f"Dataset caricato: {df.shape[0]} righe × {df.shape[1]} colonne")
print(f"Prossimo step: Notebook 02 — Data Cleaning & Feature Engineering")

Notebook 01 — Data Exploration completato.
Dataset caricato: 2240 righe × 29 colonne
Prossimo step: Notebook 02 — Data Cleaning & Feature Engineering
